We want to minimize $||Ax-b||^{2}$ w.r.t $x$
\begin{align*}
||Ax-b||^{2}= (Ax - b)^T (Ax - b)=x^T (A^T A) x - 2 b^T A x + b^T b\\
\nabla_x ||Ax-b||^{2}= 2 A^T A x - 2 A^T b\\
\end{align*}
By solving setting it to $0$ and for $x$:
\begin{align*}
A^T A x = A^T b\\
x = (A^T A)^{-1} A^T b\\
\end{align*}
In terms of linear regression $b$ is our target so it's equivalent to y. Ax is a point in column space of A, which is closest to b, so A will be matrix of inputs X and x gives us the right linear combination of column vectors of A so it's $\theta$. The closed form solution is $\theta=(X^{T}X)^{-1}X^{T}y$

In [1]:
import numpy as np
import pandas as pd

In [2]:
class LinearRegression:
    def fit(self,X,y):
        self.X=np.asarray(X)
        self.y=np.asarray(y)
        self.theta=np.linalg.inv(X.T@X)@X.T@y
    def predict(self,X):
        X=np.asarray(X.copy())
        return X@np.reshape(self.theta,(self.theta.shape[0],1))

In [3]:
df=pd.read_csv('data/melb_data.csv')

In [4]:
df.columns

Index(['Suburb', 'Address', 'Rooms', 'Type', 'Price', 'Method', 'SellerG',
       'Date', 'Distance', 'Postcode', 'Bedroom2', 'Bathroom', 'Car',
       'Landsize', 'BuildingArea', 'YearBuilt', 'CouncilArea', 'Lattitude',
       'Longtitude', 'Regionname', 'Propertycount'],
      dtype='str')

In [6]:
Sub_Ba=df.groupby('Suburb')['BuildingArea'].mean()

In [7]:
for val,sub in zip(Sub_Ba.values,Sub_Ba.index):
    df.loc[(df['BuildingArea'].isna())&(df['Suburb']==sub),'BuildingArea']=val
df=df.dropna(axis=0,subset=['BuildingArea','Car'])

In [8]:
# Filter impossible or extreme building areas/prices
df = df[(df['BuildingArea'] > 20) & (df['BuildingArea'] < 1000)]

In [9]:
X=df[['Rooms','Distance','Lattitude',
       'Longtitude','Propertycount','Type','Regionname','BuildingArea','Bedroom2', 'Bathroom', 'Car']]
y=df['Price'].astype('float64')

In [10]:
X=pd.get_dummies(X,columns=['Type','Regionname'], drop_first=True, dtype='int32')

In [11]:
X['Rooms']=X['Rooms'].astype('float64')
num_cols=X.select_dtypes(include='float64').columns

In [12]:
X.insert(0,'x0',np.ones(X.shape[0]))

In [13]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

In [14]:
X_train_std=X_train.loc[:,num_cols].std(axis=0)
X_train_mean=X_train.loc[:,num_cols].mean(axis=0)
X_train.loc[:,num_cols]=(X_train.loc[:,num_cols]-X_train_mean)/X_train_std
X_test.loc[:,num_cols]=(X_test.loc[:,num_cols]-X_train_mean)/X_train_std

y_train_log=np.log1p(y_train)
y_train_mean=y_train_log.mean()
y_train_std=y_train_log.std()
y_train_log=(y_train_log-y_train_mean)/y_train_std


In [15]:
lr=LinearRegression()
lr.fit(X_train,y_train_log)
preds_log=lr.predict(X_test)

preds_log=(preds_log*y_train_std+y_train_mean).flatten()
preds=np.expm1(preds_log)
print(f"Average error: {np.mean(np.abs(y_test.values-preds))}")

Average error: 239006.59248177748
